# SIRAP backend · RT-DETR Ecuador

Detector activo: **RT-DETRv2 R18** mediante **Transformers**, checkpoint `merve/license-plates-rtdetrv2` con licencia Apache-2.0. Los datos y métricas nuevos se mantienen separados de los resultados históricos YOLO.


In [ ]:
%pip install -q "transformers>=5.4,<6" "accelerate>=1.2,<2" "pycocotools>=2.0.8" "optuna>=4,<5" "fast-plate-ocr==1.1.0" "roboflow==1.7.1"

In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO_ROOT=Path("/content/SIRAP_USA_ECUADOR")
if not REPO_ROOT.exists(): subprocess.run(["git","clone","--depth","1","https://github.com/eduardoceballos666/SIRAP_USA_ECUADOR.git",str(REPO_ROOT)],check=True)
os.chdir(REPO_ROOT); sys.path.insert(0,str(REPO_ROOT))
COUNTRY="ecuador"
MODEL_ID="merve/license-plates-rtdetrv2"
print("Perfil:",COUNTRY,"| RT-DETRv2 R18:",MODEL_ID)


## Inferencia por perfil
El backend carga `models/ecuador/detector/rtdetr_r18` si existe; en otro caso usa el checkpoint público de matrículas. Comprueba el nombre del checkpoint antes de procesar imágenes o video.

In [ ]:
from src.model import ALPRPipeline
from src.data_processing import load_image
pipeline=ALPRPipeline.from_config()
print(pipeline.detector.name)
IMAGE_PATH=None
if IMAGE_PATH: print(pipeline.recognize(load_image(Path(IMAGE_PATH).read_bytes())).as_dict())


## Video de demostración
Muestrea un cuadro cada 2,5 segundos y guarda predicciones para revisión. La evaluación requiere una referencia temporal correspondiente al mismo video.

In [ ]:
import cv2,pandas as pd
VIDEO_PATH="/content/SIRAP_workspace/video_camiones.mp4"
OUT="/content/decisions_rtdetr_ecuador.csv"
cap=cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened(): raise FileNotFoundError(VIDEO_PATH)
fps=cap.get(cv2.CAP_PROP_FPS) or 25; step=max(1,int(fps*2.5)); rows=[]; index=0
while True:
    ok,frame=cap.read()
    if not ok: break
    if index%step==0:
        result=pipeline.recognize(cv2.cvtColor(frame,cv2.COLOR_BGR2RGB))
        rows.append({"frame_index":index,"plate_pred":result.plate,"det_confidence":result.det_confidence,
                     "ocr_confidence":result.ocr_confidence,"time_ms":result.timings_ms.get("total")})
    index+=1
cap.release(); pd.DataFrame(rows).to_csv(OUT,index=False)
print("Guardado",OUT,"cuadros analizados",len(rows))
